In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, accuracy_score, classification_report
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from xgboost import XGBClassifier

# --- Load dataset ---
file_path = r"C:\Users\Zahra\OneDrive\Documents\ITDPA QUESTION 4 ASSIGNMENT\final_cleaned_asd_dataset.xlsx"
df = pd.read_excel(file_path)

# --- Define features and target ---
X = df.drop(columns=["ID", "Class/ASD", "Predicted_Class", "Probability"])
y = df["Class/ASD"]

# --- Split dataset ---
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# --- One-hot encode categorical data (AQ-10 responses) ---
X_train_encoded = pd.get_dummies(X_train)
X_test_encoded = pd.get_dummies(X_test)
X_test_encoded = X_test_encoded.reindex(columns=X_train_encoded.columns, fill_value=0)

# --- Define Models and Colors ---
models = {
    "XGBoost": (XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=42), "RdPu"),  # pink
    "SVM": (SVC(kernel='rbf', probability=True, random_state=42), "Purples"),  # purple
    "Random Forest": (RandomForestClassifier(n_estimators=100, random_state=42), "Blues")  # blue
}

results = {}

# --- Train and Evaluate on Clean Data ---
for name, (model, color) in models.items():
    model.fit(X_train_encoded, y_train)
    y_pred = model.predict(X_test_encoded)
    acc_test = accuracy_score(y_test, y_pred)
    cm = confusion_matrix(y_test, y_pred)
    tn, fp, fn, tp = cm.ravel()

    # Training accuracy
    acc_train = accuracy_score(y_train, model.predict(X_train_encoded))
    overfit_gap = acc_train - acc_test

    results[name] = {
        "model": model,
        "color": color,
        "train_acc": acc_train,
        "test_acc": acc_test,
        "overfit_gap": overfit_gap,
        "fp": fp,
        "fn": fn,
    }

    # --- Plot Confusion Matrix ---
    plt.figure(figsize=(5.5, 4.5))
    sns.heatmap(cm, annot=True, fmt='d', cmap=color, cbar=False,
                annot_kws={"size": 12, "weight": "bold"})
    plt.title(f"Confusion Matrix – {name}", fontsize=13, fontweight='bold')
    plt.xlabel("Predicted Labels")
    plt.ylabel("Actual Labels")
    plt.xticks([0.5, 1.5], ['Non-ASD', 'ASD'])
    plt.yticks([0.5, 1.5], ['Non-ASD', 'ASD'])
    plt.tight_layout()
    plt.show()

    # --- Print Classification Report ---
    print(f"\n===== {name} - Classification Report (Clean Data) =====\n")
    print(classification_report(y_test, y_pred, target_names=['Non-ASD', 'ASD']))
    print("-" * 60)

# --- Add Noise to Test Data (AQ-10 style) ---
X_test_noisy = X_test.copy()
categories = ["Definitely Agree", "Slightly Agree", "Slightly Disagree", "Definitely Disagree"]
categorical_cols = X_test_noisy.select_dtypes(include='object').columns

np.random.seed(42)
for col in categorical_cols:
    idx = np.random.choice(X_test_noisy.index, size=int(0.05 * len(X_test_noisy)), replace=False)
    for i in idx:
        current_value = X_test_noisy.loc[i, col]
        possible_choices = [c for c in categories if c != current_value]
        X_test_noisy.loc[i, col] = np.random.choice(possible_choices)

# --- Encode noisy data ---
X_test_noisy_encoded = pd.get_dummies(X_test_noisy)
X_test_noisy_encoded = X_test_noisy_encoded.reindex(columns=X_train_encoded.columns, fill_value=0)

# --- Evaluate Models on Noisy Data ---
for name, data in results.items():
    model = data["model"]
    y_pred_noisy = model.predict(X_test_noisy_encoded)
    acc_noisy = accuracy_score(y_test, y_pred_noisy)
    results[name]["acc_noisy"] = acc_noisy
    change = ((acc_noisy - data["test_acc"]) / data["test_acc"]) * 100
    results[name]["noise_change"] = change

    # Classification report for noisy data
    print(f"\n===== {name} - Classification Report (With 5% Noise) =====\n")
    print(classification_report(y_test, y_pred_noisy, target_names=['Non-ASD', 'ASD']))
    print("-" * 60)

# --- Overfitting Analysis Bar Graph ---
plt.figure(figsize=(8,5))
models_list = list(results.keys())
train_acc = [results[m]["train_acc"] for m in models_list]
test_acc = [results[m]["test_acc"] for m in models_list]
colors = ['pink', 'purple', 'blue']

bar_width = 0.35
x = np.arange(len(models_list))

plt.bar(x, train_acc, width=bar_width, color=colors, alpha=0.7, label='Training Accuracy')
plt.bar(x + bar_width, test_acc, width=bar_width, color=['lightcoral','violet','skyblue'], alpha=0.8, label='Test Accuracy')

plt.xlabel('Model', fontsize=12, fontweight='bold')
plt.ylabel('Accuracy', fontsize=12, fontweight='bold')
plt.title('Overfitting Analysis - Training vs Test Accuracy', fontsize=14, fontweight='bold')
plt.xticks(x + bar_width/2, models_list)
plt.ylim(0, 1.1)
plt.legend()
plt.tight_layout()
plt.show()

# --- Error Analysis: False Positives vs False Negatives ---
plt.figure(figsize=(8,5))
false_pos = [results[m]["fp"] for m in models_list]
false_neg = [results[m]["fn"] for m in models_list]

plt.bar(x, false_pos, width=bar_width, color=colors, alpha=0.7, label='False Positives')
plt.bar(x + bar_width, false_neg, width=bar_width, color=['lightcoral','violet','skyblue'], alpha=0.8, label='False Negatives')

plt.xlabel('Model', fontsize=12, fontweight='bold')
plt.ylabel('Count', fontsize=12, fontweight='bold')
plt.title('Error Analysis - False Positives vs False Negatives', fontsize=14, fontweight='bold')
plt.xticks(x + bar_width/2, models_list)
plt.legend()
plt.tight_layout()
plt.show()

# --- Summary Table ---
summary = pd.DataFrame({
    "Model": models_list,
    "Test Accuracy": [results[m]["test_acc"] for m in models_list],
    "Training Accuracy": [results[m]["train_acc"] for m in models_list],
    "Overfitting Gap": [results[m]["overfit_gap"] for m in models_list],
    "False Positives": [results[m]["fp"] for m in models_list],
    "False Negatives": [results[m]["fn"] for m in models_list],
    "Noise Sensitivity (%)": [f"{results[m]['noise_change']:.1f}%" for m in models_list]
})

print("\n=== Final Model Performance Summary ===\n")
print(summary.to_string(index=False))
